In [9]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [10]:
#creating session

spark = (( SparkSession 
            .builder
            .appName('dim_transaction')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
            
        ))

In [11]:
tmp_transactionframe = (( spark
                            .read
                                .format('jdbc')
                                .option('url', mssqlsource)
                                .option('driver', mssqldriver)
                                .option('user', mssqluser)
                                .option('password', mssqlpwd)
                                .option('query', 
                                    
                                        """ 
                                        select
                                        
                                            transactiontypeid,
                                            transactiontypename
                                            
                                        from application.transactiontypes
                                        """)
                        )).load()

In [12]:
#normalize columns name

transactionframe = (( tmp_transactionframe
                        .withColumnRenamed('transactiontypename', 'transaction_name')
                    ))

In [13]:
(( transactionframe
    .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('driver', mssqldriver)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('dbtable', 'dimension.transactiontype')
    .save(mode= 'overwrite')
))